In [ ]:
import pandas as pd
import json
import time
from pandas import json_normalize
from vk_api import ApiError #импортируем ошибки API
import vk_api

In [ ]:
#ключ доступа пользователя
user_token = input("Введите токен пользователя: ")

In [ ]:
#название города для поиска пользователей
HometownName = input("Введите название города: ")

In [ ]:
version = 5.199 #версия API «ВКонтакте»
#инициализируем сессию в ВК при помощи ключа доступа пользователя
session = vk_api.VkApi(token=user_token)
vk=session.get_api()

In [ ]:
#запрос на поиск городов по названию
dfCities=pd.DataFrame.from_records(vk.database.getCities(q=HometownName, need_all=1)['items'])
#достаем идентификатор первого города
CitiesID = dfCities['id'][0]
#ищем пользователей по идентификатору города
UserByGeoposition=pd.DataFrame.from_records(vk.users.search(count=1000, city=CitiesID, sex=0, online=0, has_photo=0)['items'])

In [ ]:
#функция определения открытых и незаблокированных страниц пользователей
def GetVerifiedUserID(users_l):
    ver_users_l=[] #список идентификаторов открытых и незаблокированных пользователей
    for CheckUserID in users_l:
        time.sleep(0.35)
        user_info=pd.DataFrame.from_records(vk.users.get(user_ids=CheckUserID)) #получаем информацию о пользователе и записываем её в DataFrame
        #если он закрыт или декативирован (удален, забанен), то пререходим к следующему пользователю, иначе вносим в возвращаемый список открытых пользователей 
        if (user_info.loc[0,'is_closed']==True) or ('deactivated' in user_info.columns):
            continue
        else:
            ver_users_l.append(CheckUserID)
    return(ver_users_l)

In [ ]:
#функция сбора постов и идентификаторов подписчиков пользователей
def TakePostsFollowers(verified_users_l):
    res_users_text = [] #список кортежей результата
    #для каждого идентификатора пользователя
    for VerifiedUserID in verified_users_l:
        time.sleep(0.35)
        #пробуем собрать 100 записей со стены
        try:
            df_wall = vk.wall.get(domain=VerifiedUserID, offset=0, count=100)
            df_wall = json_normalize(df_wall['items'])
            
            #если есть столбец 'текст'
            if 'text' in df_wall.columns:
                post_text = df_wall['text'].to_list() #перевод столбец в список 
                if post_text!=[]:
                    UserPosts_text =' '.join(post_text) #объединяем элементы столбца в строку
    
                    #задаем запрос на получение 1000 подписчиков пользователя
                    time.sleep(0.35)
                    followers=vk.users.getFollowers(user_id=VerifiedUserID, offset=0, count=1000) 
                    followers_l=followers['items']
                    
                    tuple_users_text = (VerifiedUserID, UserPosts_text, followers_l) #создаем кортеж результата в котором id пользователя, текст его постов и подписчики
                    res_users_text.append(tuple_users_text) #вносим кортеж результата в список кортежей
                else:
                    tuple_users_text = (VerifiedUserID, '', []) 
                    res_users_text.append(tuple_users_text)
            else:
                tuple_users_text = (VerifiedUserID, '', []) 
                res_users_text.append(tuple_users_text)
                #если нет, записываем в кортеж идентификатор пользователя, пустую строку в качестве текста постов, пустой список в качестве списка подписчиков
            
        except ApiError:
            tuple_users_text = (VerifiedUserID, '', [])
            res_users_text.append(tuple_users_text)
            #если возникла ошибка API, записываем в кортеж идентификатор пользователя, пустую строку в качестве текста постов и пустой список в качестве списка подписчиков
    return(res_users_text)

In [ ]:
#функция для нахождения подписчиков из города
def GetFollowersInHometown(user_post_follower):
    res_get_followers = [] #список кортежей результата
	for x in range(user_post_follower.shape[0]):
        followers_l = user_post_follower['followers'][x]
		for posible_follower in followers_l:
            if posible_follower in user_post_follower['id'].to_list():
				tuple_followers=(user_post_follower['id'][x], posible_follower, 1)
				res_get_followers.append(tuple_followers)
	return(res_get_followers)

In [ ]:
users_id = [] #идентификаторы пользователей
users_text = [] #тексты постов пользователей
batchs_ids = [] #список пакетов
batch_of_ids = [] #пакет идентификаторов пользователей

In [ ]:
#разделяем идентификаторы по пакетам
for RowUserID in range(len(UserByGeoposition)):
    if len(batch_of_ids)==25:
        batchs_ids.append(batch_of_ids)
		batch_of_ids = [UserByGeoposition.loc[RowUserID, 'id']]
        #если размер пакета, к которому принадлежит пользователь, равен 25, вносим этот пакет в список пакетов, инициализируем пакет текущим идентификатором пользователя 
	elif (RowUserID==len(UserByGeoposition)):
        batch_of_ids.append(UserByGeoposition.loc[RowUserID, 'id'])
		batchs_ids.append(batch_of_ids)
        #если идентификатор текущего пользователя последний, вносим его в пакет и записываем получившийся пакет в список пакетов 
	else:
		batch_of_ids.append(UserByGeoposition.loc[RowUserID, 'id'])
        #иначе добавляем идентификатор пользователя в пакет

In [ ]:
#проходимся по всем пакетам, сбор текстов постов и подписчиков
for package_of_ids in batchs_ids:
	VerifiedUsersID = GetVerifiedUserID(package_of_ids)
	resTakePosts = TakePosts(VerifiedUsersID)
    
	res_users = [r_u[0] for r_u in resTakePosts]
	res_text = [r_u[1] for r_u in resTakePosts]
	res_followers = [r_u[2] for r_u in resTakePostsFollowers]
    
	users_id.extend(res_users)
	users_text.extend(res_text)
	users_followers.extend(res_followers)

In [ ]:
user_posts_followers = pd.DataFrame(
    {'id': users_id,
     'post': users_text,
     'followers': users_followers})
user_posts_followers

In [ ]:
drop_indices = user_posts_followers.index[user_posts_followers['post'] == ' '].tolist()
UserPostsFollowers = user_posts_followers.drop(drop_indices, axis=0)
drop_indices = user_posts_followers.index[user_posts_followers['post'] == ''].tolist()
UserPostsFollowers = UserPostsFollowers.drop(drop_indices, axis=0)
UserPostsFollowers.reset_index(drop=True, inplace=True)

In [ ]:
#находим подписки пользователей из города друг на друга
resGetFollowers = GetFollowersInHometown(UserPostsFollowers)
if (len(resGetFollowers)!=0):
	UsersFollowersInHometown = pd.DataFrame(resGetFollowers, columns=['user_id', 'follower', 'weight'])
	UsersFollowersInHometown.to_excel('подписки пользователей из города на друг друга.xlsx', index=False)

In [ ]:
df_user_posts = pd.DataFrame({'id': UserPostsFollowers['id'], 'post': UserPostsFollowers['post']})
df_user_posts.to_excel('пользователи из города.xlsx', index=False)